# TP Théorie des jeux : dominance stricte et dominance faible

## Plan du notebook

| Partie | Contenu | Consigne du TP |
|---|---|---|
| 1 | Représentation d'un jeu et saisie (clavier, fichier, aléatoire) | R1 |
| 2 | Affichage des matrices | R2 |
| 3 | Test de dominance (stricte ou faible) | R3, R4 |
| 4 | Élimination itérée avec trace et résultat final | R5, R6 |
| 5 | Jeux de test (exemples 1 à 5) | Section 5 du TP |
| 6 | Fichier, génération aléatoire, saisie clavier | R1 |
| 7 | Tableau récapitulatif des tests | Section 5 du TP |

In [1]:
import itertools   # pour parcourir toutes les combinaisons de stratégies
import random      # pour la génération aléatoire
import json        # pour lire et écrire un jeu dans un fichier

### Trois façons de fournir un jeu

1. **Au clavier** : `saisie_clavier()`.
2. **Par fichier** : `lire_fichier(chemin)` (et `sauvegarder_fichier` pour créer le fichier).
3. **Aléatoirement** : `jeu_aleatoire(tailles)`.

On ajoute aussi `jeu_depuis_matrice(...)`, une aide pour écrire facilement les jeux à 2 joueurs.

In [ ]:
def jeu_depuis_matrice(lignes, colonnes, matrice):
    # Construit (noms, gains) à partir d'une matrice écrite à la main (2 joueurs).
    gains = {}
    for x in range(len(lignes)):
        for y in range(len(colonnes)):
            gains[(lignes[x], colonnes[y])] = matrice[x][y]
    return [lignes, colonnes], gains

def nombre(texte):
    # Transforme un texte en nombre.
    x = float(texte)
    if x == int(x):
        return int(x)
    return x


def saisie_clavier():
    # 1) nombre de joueurs et de stratégies, 2) gains de chaque profil.
    n = int(input("Nombre de joueurs : "))
    noms = []
    for i in range(n):
        k = int(input(f"Nombre de stratégies du joueur {i+1} : "))
        # stratégies nommées A1, A2, ... pour le joueur 1, B1, B2, ... pour le joueur 2
        noms.append([f"{chr(65 + i)}{j + 1}" for j in range(k)])

    gains = {}
    for profil in itertools.product(*noms):
        texte = input(f"Gains pour {profil} ({n} nombres séparés par des espaces) : ")
        gains[profil] = tuple(nombre(t) for t in texte.split())
    return noms, gains


def jeu_aleatoire(tailles, gain_max=10):
    # tailles = [k1, k2, ..., kn] : nombre de stratégies de chaque joueur.
    noms = []
    for i in range(len(tailles)):
        noms.append([f"{chr(65 + i)}{j + 1}" for j in range(tailles[i])])

    gains = {}
    for profil in itertools.product(*noms):
        gains[profil] = tuple(random.randint(0, gain_max) for _ in tailles)
    return noms, gains


def sauvegarder_fichier(chemin, noms, gains):
    # Chaque ligne du fichier : les stratégies du profil, puis les gains.
    lignes = [list(profil) + list(u) for profil, u in gains.items()]
    with open(chemin, "w", encoding="utf-8") as f:
        json.dump({"joueurs": noms, "gains": lignes}, f, ensure_ascii=False)


def lire_fichier(chemin):
    with open(chemin, encoding="utf-8") as f:
        donnees = json.load(f)
    noms = donnees["joueurs"]
    n = len(noms)
    gains = {}
    for ligne in donnees["gains"]:
        gains[tuple(ligne[:n])] = tuple(ligne[n:])
    return noms, gains

## Partie 2 : affichage des matrices (R2)

* **2 joueurs** : une seule matrice, joueur 1 en lignes, joueur 2 en colonnes, chaque case contient le vecteur de gains.
* **3 joueurs ou plus** : joueurs 1 et 2 en lignes et colonnes, et **une matrice par combinaison** de stratégies des joueurs 3 à n,
  clairement étiquetée.

On affiche seulement les stratégies **encore actives** (liste `actives`), c'est ce qui permet de montrer le jeu après chaque suppression.

In [3]:
def afficher_matrice(gains, actives, fixes=()):
    # fixes = stratégies choisies par les joueurs 3, 4, ... (vide pour 2 joueurs)
    lignes = actives[0]
    colonnes = actives[1]
    largeur = 14

    print("J1\\J2".ljust(largeur) + "".join(c.center(largeur) for c in colonnes))
    for l in lignes:
        texte = l.ljust(largeur)
        for c in colonnes:
            texte += str(gains[(l, c) + tuple(fixes)]).center(largeur)
        print(texte)


def afficher_jeu(gains, actives):
    if len(actives) == 2:
        afficher_matrice(gains, actives)
    else:
        # une matrice pour chaque combinaison de stratégies des joueurs 3 à n
        for fixes in itertools.product(*actives[2:]):
            etiquette = ", ".join(f"J{k + 3}={s}" for k, s in enumerate(fixes))
            print(f"--- {etiquette} ---")
            afficher_matrice(gains, actives, fixes)
            print()

## Partie 3 : test de dominance (R3 et R4)

La fonction `domine(gains, actives, i, b, a, mode)` répond à la question :
**« la stratégie b du joueur i domine-t-elle la stratégie a ? »**

Principe :

1. On parcourt **toutes les combinaisons** des stratégies encore actives des **autres** joueurs.
2. Pour chacune, on compare le gain du joueur *i* avec `b` et avec `a` (les autres joueurs ne changent pas).
3. On retient trois informations :
   * `b` est-elle strictement meilleure **partout** ? (dominance stricte)
   * `b` est-elle meilleure ou égale **partout** ?
   * `b` est-elle strictement meilleure **au moins une fois** ?
4. **Mode strict** : il faut « strictement meilleure partout ».
   **Mode faible** : il faut « meilleure ou égale partout » **et** « strictement meilleure au moins une fois ».

Si deux stratégies ont les mêmes gains, `b` n'est jamais strictement meilleure : aucune des deux ne domine l'autre, comme demandé.

In [ ]:
def domine(gains, actives, i, b, a, mode):
    # True si la stratégie b du joueur i domine la stratégie a (mode = "strict" ou "faible")
    autres_joueurs = [actives[j] for j in range(len(actives)) if j != i]

    strict_partout = True       # b > a pour toutes les combinaisons 
    superieur_partout = True    # b >= a pour toutes les combinaisons 
    meilleur_une_fois = False   # b > a pour au moins une combinaison 

    for autres in itertools.product(*autres_joueurs):
        # on insère a (ou b) à la place du joueur i dans le profil
        ua = gains[autres[:i] + (a,) + autres[i:]][i]
        ub = gains[autres[:i] + (b,) + autres[i:]][i]

        if ub <= ua:
            strict_partout = False
        if ub < ua:
            superieur_partout = False
        if ub > ua:
            meilleur_une_fois = True

    if mode == "strict":
        return strict_partout
    return superieur_partout and meilleur_une_fois

## Partie 4 : élimination itérée avec trace (R5 et R6)

### Principe

1. On part du jeu complet : `actives` = toutes les stratégies de chaque joueur.
2. `trouver_elimination` cherche **une** stratégie dominée (joueur par joueur, dans l'ordre choisi).
3. On la supprime de `actives`, on affiche l'étape et le jeu obtenu.
4. On recommence, jusqu'à ce qu'il n'y ait plus rien à supprimer.
5. On affiche le jeu réduit et on indique s'il reste **un seul profil**.

### Choix importants

* On supprime **une seule stratégie à la fois** : la trace est ainsi plus claire (une ligne par étape).
* On parcourt les joueurs dans l'ordre donné par `ordre_joueurs` (par défaut J1, J2, ..., Jn).
  Avec la **dominance stricte**, l'ordre ne change pas le résultat final.
  Avec la **dominance faible**, il peut le changer (exemple 3), donc **le programme affiche l'ordre suivi**.
* Si plusieurs stratégies en dominent une autre, on indique la **première trouvée** dans l'ordre des stratégies.

In [5]:
def trouver_elimination(gains, actives, mode, ordre_joueurs):
    # Cherche UNE stratégie dominée. Renvoie (joueur, stratégie dominée, stratégie dominante)
    # ou None s'il n'y en a plus.
    for i in ordre_joueurs:
        for a in actives[i]:
            for b in actives[i]:
                if a != b and domine(gains, actives, i, b, a, mode):
                    return i, a, b
    return None


def afficher_resultat(gains, actives, nb_etapes):
    # R6 : jeu réduit et indication "un seul profil" ou non
    print("=" * 50)
    if nb_etapes == 0:
        print("Aucune élimination n'est possible : le jeu reste inchangé.")
    print("Jeu réduit final :")
    afficher_jeu(gains, actives)

    if all(len(s) == 1 for s in actives):
        profil = tuple(s[0] for s in actives)
        print(f"Il reste un seul profil : {profil} avec les gains {gains[profil]}")
    else:
        for i in range(len(actives)):
            print(f"Stratégies restantes du joueur {i + 1} : {actives[i]}")
        print("Il reste plusieurs profils : le jeu n'est pas réduit à un seul profil.")


def elimination_iteree(gains, noms, mode="strict", ordre_joueurs=None, trace=True):
    n = len(noms)
    if ordre_joueurs is None:
        ordre_joueurs = list(range(n))            # J1, J2, ..., Jn
    adverbe = {"strict": "strictement", "faible": "faiblement"}[mode]
    actives = [list(s) for s in noms]             # copie : on ne modifie pas noms

    if trace:
        print(f"Mode : dominance {mode}")
        print("Ordre de parcours des joueurs :", " puis ".join(f"J{i + 1}" for i in ordre_joueurs))
        print()
        print("Jeu de départ :")
        afficher_jeu(gains, actives)
        print()

    nb_etapes = 0
    while True:
        resultat = trouver_elimination(gains, actives, mode, ordre_joueurs)
        if resultat is None:
            break                                  # plus rien à supprimer
        i, a, b = resultat
        actives[i].remove(a)
        nb_etapes += 1
        if trace:
            print(f"Étape {nb_etapes} : le joueur {i + 1} supprime {a}, {adverbe} dominée par {b}")
            afficher_jeu(gains, actives)
            print()

    if trace:
        afficher_resultat(gains, actives, nb_etapes)
    return actives

## Partie 5 : les jeux de test du TP

On définit d'abord les cinq jeux de l'énoncé.

In [ ]:
#Exemple 1 : élimination possible jusqu'à 1x1 (dominance stricte)
noms1, gains1 = jeu_depuis_matrice(
    ["H", "M", "B"], ["G", "C", "D"],
    [[(3, 1), (2, 3), (0, 0)],
     [(1, 2), (1, 2), (5, 1)],
     [(2, 2), (0, 1), (1, 0)]])

#Exemple 2 : Pierre-Feuille-Ciseaux (aucune élimination)
noms2, gains2 = jeu_depuis_matrice(
    ["Pierre", "Feuille", "Ciseaux"], ["Pierre", "Feuille", "Ciseaux"],
    [[(0, 0), (-1, 1), (1, -1)],
     [(1, -1), (0, 0), (-1, 1)],
     [(-1, 1), (1, -1), (0, 0)]])

#Exemple 3 : dominance faible et ordre d'élimination
noms3, gains3 = jeu_depuis_matrice(
    ["H", "B"], ["G", "M", "D"],
    [[(2, 1), (2, 1), (2, 0)],
     [(2, 1), (1, 0), (2, 0)]])

#Exemple 4 : 3 joueurs (contribuer C ou ne pas contribuer N)
def gains_contribution(profil):
    nb_C = profil.count("C")
    resultat = []
    for choix in profil:
        if choix == "C":
            resultat.append(2 * nb_C - 3)
        else:
            resultat.append(2 * nb_C)
    return tuple(resultat)

noms4 = [["C", "N"], ["C", "N"], ["C", "N"]]
gains4 = {profil: gains_contribution(profil) for profil in itertools.product(*noms4)}

#Exemple 5 : exercice 4 de la Série 1
noms5, gains5 = jeu_depuis_matrice(
    ["A1", "A2", "A3"], ["B1", "B2", "B3"],
    [[(10, 10), (4, 9), (4, 11)],
     [(5, 6), (8, 3), (3, 4)],
     [(4, 0), (8, 1), (5, 3)]])

print("Les 5 jeux sont définis.")
print("Exemple 4, vérification de la table de l'énoncé :")
for profil in itertools.product(*noms4):
    print("  ", profil, "->", gains4[profil])

Les 5 jeux sont définis.
Exemple 4, vérification de la table de l'énoncé :
   ('C', 'C', 'C') -> (3, 3, 3)
   ('C', 'C', 'N') -> (1, 1, 4)
   ('C', 'N', 'C') -> (1, 4, 1)
   ('C', 'N', 'N') -> (-1, 2, 2)
   ('N', 'C', 'C') -> (4, 1, 1)
   ('N', 'C', 'N') -> (2, -1, 2)
   ('N', 'N', 'C') -> (2, 2, -1)
   ('N', 'N', 'N') -> (0, 0, 0)


### Exemple 1 : dominance stricte

Résultat attendu : le seul profil **(H, C)** avec les gains **(2, 3)**.

In [7]:
res = elimination_iteree(gains1, noms1, mode="strict")

Mode : dominance strict
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2               G             C             D       
H                 (3, 1)        (2, 3)        (0, 0)    
M                 (1, 2)        (1, 2)        (5, 1)    
B                 (2, 2)        (0, 1)        (1, 0)    

Étape 1 : le joueur 2 supprime D, strictement dominée par G
J1\J2               G             C       
H                 (3, 1)        (2, 3)    
M                 (1, 2)        (1, 2)    
B                 (2, 2)        (0, 1)    

Étape 2 : le joueur 1 supprime M, strictement dominée par H
J1\J2               G             C       
H                 (3, 1)        (2, 3)    
B                 (2, 2)        (0, 1)    

Étape 3 : le joueur 1 supprime B, strictement dominée par H
J1\J2               G             C       
H                 (3, 1)        (2, 3)    

Étape 4 : le joueur 2 supprime G, strictement dominée par C
J1\J2               C       
H                 (2, 3)    

### Exemple 2 : Pierre-Feuille-Ciseaux

Résultat attendu : **aucune élimination**, la matrice 3x3 reste inchangée, en strict **et** en faible.

In [8]:
res = elimination_iteree(gains2, noms2, mode="strict")

Mode : dominance strict
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2             Pierre       Feuille       Ciseaux    
Pierre            (0, 0)       (-1, 1)       (1, -1)    
Feuille          (1, -1)        (0, 0)       (-1, 1)    
Ciseaux          (-1, 1)       (1, -1)        (0, 0)    

Aucune élimination n'est possible : le jeu reste inchangé.
Jeu réduit final :
J1\J2             Pierre       Feuille       Ciseaux    
Pierre            (0, 0)       (-1, 1)       (1, -1)    
Feuille          (1, -1)        (0, 0)       (-1, 1)    
Ciseaux          (-1, 1)       (1, -1)        (0, 0)    
Stratégies restantes du joueur 1 : ['Pierre', 'Feuille', 'Ciseaux']
Stratégies restantes du joueur 2 : ['Pierre', 'Feuille', 'Ciseaux']
Il reste plusieurs profils : le jeu n'est pas réduit à un seul profil.


In [9]:
res = elimination_iteree(gains2, noms2, mode="faible")

Mode : dominance faible
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2             Pierre       Feuille       Ciseaux    
Pierre            (0, 0)       (-1, 1)       (1, -1)    
Feuille          (1, -1)        (0, 0)       (-1, 1)    
Ciseaux          (-1, 1)       (1, -1)        (0, 0)    

Aucune élimination n'est possible : le jeu reste inchangé.
Jeu réduit final :
J1\J2             Pierre       Feuille       Ciseaux    
Pierre            (0, 0)       (-1, 1)       (1, -1)    
Feuille          (1, -1)        (0, 0)       (-1, 1)    
Ciseaux          (-1, 1)       (1, -1)        (0, 0)    
Stratégies restantes du joueur 1 : ['Pierre', 'Feuille', 'Ciseaux']
Stratégies restantes du joueur 2 : ['Pierre', 'Feuille', 'Ciseaux']
Il reste plusieurs profils : le jeu n'est pas réduit à un seul profil.


### Exemple 3 : dominance stricte

Résultat attendu : `D` est supprimée, il reste la matrice 2x2 avec les colonnes `G` et `M`.

In [10]:
res = elimination_iteree(gains3, noms3, mode="strict")

Mode : dominance strict
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2               G             M             D       
H                 (2, 1)        (2, 1)        (2, 0)    
B                 (2, 1)        (1, 0)        (2, 0)    

Étape 1 : le joueur 2 supprime D, strictement dominée par G
J1\J2               G             M       
H                 (2, 1)        (2, 1)    
B                 (2, 1)        (1, 0)    

Jeu réduit final :
J1\J2               G             M       
H                 (2, 1)        (2, 1)    
B                 (2, 1)        (1, 0)    
Stratégies restantes du joueur 1 : ['H', 'B']
Stratégies restantes du joueur 2 : ['G', 'M']
Il reste plusieurs profils : le jeu n'est pas réduit à un seul profil.


### Exemple 3 : dominance faible, le résultat dépend de l'ordre

* **Ordre J1 puis J2** : J1 supprime `B` (dominée faiblement par `H`), puis J2 supprime `D`. Résultat attendu : `H × {G, M}`.
* **Ordre J2 puis J1** : J2 supprime `M` (dominée faiblement par `G`), puis `D`. Résultat attendu : `{H, B} × G`.

Le programme affiche l'ordre suivi, comme demandé.

In [11]:
res = elimination_iteree(gains3, noms3, mode="faible", ordre_joueurs=[0, 1])

Mode : dominance faible
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2               G             M             D       
H                 (2, 1)        (2, 1)        (2, 0)    
B                 (2, 1)        (1, 0)        (2, 0)    

Étape 1 : le joueur 1 supprime B, faiblement dominée par H
J1\J2               G             M             D       
H                 (2, 1)        (2, 1)        (2, 0)    

Étape 2 : le joueur 2 supprime D, faiblement dominée par G
J1\J2               G             M       
H                 (2, 1)        (2, 1)    

Jeu réduit final :
J1\J2               G             M       
H                 (2, 1)        (2, 1)    
Stratégies restantes du joueur 1 : ['H']
Stratégies restantes du joueur 2 : ['G', 'M']
Il reste plusieurs profils : le jeu n'est pas réduit à un seul profil.


In [12]:
res = elimination_iteree(gains3, noms3, mode="faible", ordre_joueurs=[1, 0])

Mode : dominance faible
Ordre de parcours des joueurs : J2 puis J1

Jeu de départ :
J1\J2               G             M             D       
H                 (2, 1)        (2, 1)        (2, 0)    
B                 (2, 1)        (1, 0)        (2, 0)    

Étape 1 : le joueur 2 supprime M, faiblement dominée par G
J1\J2               G             D       
H                 (2, 1)        (2, 0)    
B                 (2, 1)        (2, 0)    

Étape 2 : le joueur 2 supprime D, faiblement dominée par G
J1\J2               G       
H                 (2, 1)    
B                 (2, 1)    

Jeu réduit final :
J1\J2               G       
H                 (2, 1)    
B                 (2, 1)    
Stratégies restantes du joueur 1 : ['H', 'B']
Stratégies restantes du joueur 2 : ['G']
Il reste plusieurs profils : le jeu n'est pas réduit à un seul profil.


### Exemple 4 : trois joueurs (dominance stricte)

Pour chaque joueur, `N` domine strictement `C`. Résultat attendu : le seul profil **(N, N, N)** avec les gains **(0, 0, 0)**.
Ici le jeu est affiché avec **une matrice par stratégie du joueur 3** (`J3=C` et `J3=N`).

In [13]:
res = elimination_iteree(gains4, noms4, mode="strict")

Mode : dominance strict
Ordre de parcours des joueurs : J1 puis J2 puis J3

Jeu de départ :
--- J3=C ---
J1\J2               C             N       
C               (3, 3, 3)     (1, 4, 1)   
N               (4, 1, 1)     (2, 2, -1)  

--- J3=N ---
J1\J2               C             N       
C               (1, 1, 4)     (-1, 2, 2)  
N               (2, -1, 2)    (0, 0, 0)   


Étape 1 : le joueur 1 supprime C, strictement dominée par N
--- J3=C ---
J1\J2               C             N       
N               (4, 1, 1)     (2, 2, -1)  

--- J3=N ---
J1\J2               C             N       
N               (2, -1, 2)    (0, 0, 0)   


Étape 2 : le joueur 2 supprime C, strictement dominée par N
--- J3=C ---
J1\J2               N       
N               (2, 2, -1)  

--- J3=N ---
J1\J2               N       
N               (0, 0, 0)   


Étape 3 : le joueur 3 supprime C, strictement dominée par N
--- J3=N ---
J1\J2               N       
N               (0, 0, 0)   


Jeu réduit final :
---

### Exemple 5 : exercice 4 de la Série 1

Résultat attendu, en strict **et** en faible : le seul profil **(A3, B3)** avec les gains **(5, 3)**.

In [14]:
res = elimination_iteree(gains5, noms5, mode="strict")

Mode : dominance strict
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2               B1            B2            B3      
A1               (10, 10)       (4, 9)       (4, 11)    
A2                (5, 6)        (8, 3)        (3, 4)    
A3                (4, 0)        (8, 1)        (5, 3)    

Étape 1 : le joueur 2 supprime B2, strictement dominée par B3
J1\J2               B1            B3      
A1               (10, 10)      (4, 11)    
A2                (5, 6)        (3, 4)    
A3                (4, 0)        (5, 3)    

Étape 2 : le joueur 1 supprime A2, strictement dominée par A1
J1\J2               B1            B3      
A1               (10, 10)      (4, 11)    
A3                (4, 0)        (5, 3)    

Étape 3 : le joueur 2 supprime B1, strictement dominée par B3
J1\J2               B3      
A1               (4, 11)    
A3                (5, 3)    

Étape 4 : le joueur 1 supprime A1, strictement dominée par A3
J1\J2               B3      
A3                (

In [15]:
res = elimination_iteree(gains5, noms5, mode="faible")

Mode : dominance faible
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2               B1            B2            B3      
A1               (10, 10)       (4, 9)       (4, 11)    
A2                (5, 6)        (8, 3)        (3, 4)    
A3                (4, 0)        (8, 1)        (5, 3)    

Étape 1 : le joueur 2 supprime B2, faiblement dominée par B3
J1\J2               B1            B3      
A1               (10, 10)      (4, 11)    
A2                (5, 6)        (3, 4)    
A3                (4, 0)        (5, 3)    

Étape 2 : le joueur 1 supprime A2, faiblement dominée par A1
J1\J2               B1            B3      
A1               (10, 10)      (4, 11)    
A3                (4, 0)        (5, 3)    

Étape 3 : le joueur 2 supprime B1, faiblement dominée par B3
J1\J2               B3      
A1               (4, 11)    
A3                (5, 3)    

Étape 4 : le joueur 1 supprime A1, faiblement dominée par A3
J1\J2               B3      
A3                (5, 3

## Partie 6 : fichier, génération aléatoire, saisie clavier (R1)

### Par fichier

On sauvegarde l'exemple 1 dans `exemple1.json`, on le relit, et on vérifie qu'on retrouve le même jeu et le même résultat.

Format du fichier (JSON) : la liste des stratégies de chaque joueur, puis une ligne par profil
(les stratégies du profil suivies des gains).

In [16]:
sauvegarder_fichier("exemple1.json", noms1, gains1)
noms_f, gains_f = lire_fichier("exemple1.json")

print("Même jeu après lecture du fichier ?", noms_f == noms1 and gains_f == gains1)
res = elimination_iteree(gains_f, noms_f, mode="strict", trace=False)
print("Résultat sur le jeu lu depuis le fichier :", res)

Même jeu après lecture du fichier ? True
Résultat sur le jeu lu depuis le fichier : [['H'], ['C']]


### Par génération aléatoire

`jeu_aleatoire([3, 3])` crée un jeu à 2 joueurs avec 3 stratégies chacun, gains entiers entre 0 et 10.
La graine `random.seed(...)` rend l'exemple reproductible.
Les tailles peuvent être différentes (`[3, 2]`) et le nombre de joueurs quelconque (`[2, 2, 2]`).

In [17]:
random.seed(7)
noms_a, gains_a = jeu_aleatoire([3, 3])
res = elimination_iteree(gains_a, noms_a, mode="strict")

Mode : dominance strict
Ordre de parcours des joueurs : J1 puis J2

Jeu de départ :
J1\J2               B1            B2            B3      
A1                (5, 2)       (6, 10)        (0, 1)    
A2                (8, 1)        (5, 9)        (0, 8)    
A3                (3, 0)        (1, 6)        (6, 1)    

Étape 1 : le joueur 2 supprime B1, strictement dominée par B2
J1\J2               B2            B3      
A1               (6, 10)        (0, 1)    
A2                (5, 9)        (0, 8)    
A3                (1, 6)        (6, 1)    

Étape 2 : le joueur 2 supprime B3, strictement dominée par B2
J1\J2               B2      
A1               (6, 10)    
A2                (5, 9)    
A3                (1, 6)    

Étape 3 : le joueur 1 supprime A2, strictement dominée par A1
J1\J2               B2      
A1               (6, 10)    
A3                (1, 6)    

Étape 4 : le joueur 1 supprime A3, strictement dominée par A1
J1\J2               B2      
A1               (6, 10)    

Je

In [18]:
# jeu aléatoire à 3 joueurs avec des nombres de stratégies différents
random.seed(3)
noms_b, gains_b = jeu_aleatoire([2, 3, 2])
res = elimination_iteree(gains_b, noms_b, mode="faible")

Mode : dominance faible
Ordre de parcours des joueurs : J1 puis J2 puis J3

Jeu de départ :
--- J3=C1 ---
J1\J2               B1            B2            B3      
A1              (3, 9, 8)     (7, 10, 9)    (7, 4, 8)   
A2              (8, 8, 7)     (3, 10, 2)    (10, 1, 2)  

--- J3=C2 ---
J1\J2               B1            B2            B3      
A1              (2, 5, 9)     (1, 9, 0)     (3, 3, 7)   
A2              (6, 10, 2)    (8, 6, 0)     (9, 0, 4)   


Étape 1 : le joueur 2 supprime B3, faiblement dominée par B1
--- J3=C1 ---
J1\J2               B1            B2      
A1              (3, 9, 8)     (7, 10, 9)  
A2              (8, 8, 7)     (3, 10, 2)  

--- J3=C2 ---
J1\J2               B1            B2      
A1              (2, 5, 9)     (1, 9, 0)   
A2              (6, 10, 2)    (8, 6, 0)   


Jeu réduit final :
--- J3=C1 ---
J1\J2               B1            B2      
A1              (3, 9, 8)     (7, 10, 9)  
A2              (8, 8, 7)     (3, 10, 2)  

--- J3=C2 ---
J1\J2   

### Par saisie au clavier et choix du mode

Le TP demande que l'utilisateur **choisisse le mode (strict ou faible) avant l'exécution** : c'est le rôle de `choisir_mode()`.
Ces cellules demandent des saisies, elles sont donc **en commentaire** : enlève les `#` pour les essayer.

In [ ]:
def choisir_mode():
    mode = ""
    while mode not in ("strict", "faible"):
        mode = input("Mode de dominance (strict / faible) : ").strip().lower()
    return mode


## Partie 7 : tableau récapitulatif des tests

On relance tous les jeux de test sans la trace et on compare avec les résultats attendus de l'énoncé.

In [20]:
def verifier(nom, noms, gains, mode, acceptes, ordre=None):
    # acceptes = liste des résultats acceptables (cas de l'exemple 3 faible)
    actives = elimination_iteree(gains, noms, mode, ordre, trace=False)
    etat = "OK" if actives in acceptes else "ERREUR"
    print(f"{nom:<26} {mode:<7} {str(actives):<72} {etat}")


print(f"{'Jeu':<26} {'Mode':<7} {'Jeu réduit (J1, J2, ...)':<72} Test")
print("-" * 114)
verifier("Exemple 1", noms1, gains1, "strict", [[["H"], ["C"]]])
verifier("Exemple 2", noms2, gains2, "strict", [noms2])
verifier("Exemple 2", noms2, gains2, "faible", [noms2])
verifier("Exemple 3", noms3, gains3, "strict", [[["H", "B"], ["G", "M"]]])
verifier("Exemple 3 (ordre J1, J2)", noms3, gains3, "faible", [[["H"], ["G", "M"]]], [0, 1])
verifier("Exemple 3 (ordre J2, J1)", noms3, gains3, "faible", [[["H", "B"], ["G"]]], [1, 0])
verifier("Exemple 4 (3 joueurs)", noms4, gains4, "strict", [[["N"], ["N"], ["N"]]])
verifier("Exemple 5", noms5, gains5, "strict", [[["A3"], ["B3"]]])
verifier("Exemple 5", noms5, gains5, "faible", [[["A3"], ["B3"]]])

Jeu                        Mode    Jeu réduit (J1, J2, ...)                                                 Test
------------------------------------------------------------------------------------------------------------------
Exemple 1                  strict  [['H'], ['C']]                                                           OK
Exemple 2                  strict  [['Pierre', 'Feuille', 'Ciseaux'], ['Pierre', 'Feuille', 'Ciseaux']]     OK
Exemple 2                  faible  [['Pierre', 'Feuille', 'Ciseaux'], ['Pierre', 'Feuille', 'Ciseaux']]     OK
Exemple 3                  strict  [['H', 'B'], ['G', 'M']]                                                 OK
Exemple 3 (ordre J1, J2)   faible  [['H'], ['G', 'M']]                                                      OK
Exemple 3 (ordre J2, J1)   faible  [['H', 'B'], ['G']]                                                      OK
Exemple 4 (3 joueurs)      strict  [['N'], ['N'], ['N']]                                                  